In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.4/756.4 kB 9.8 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many

# Load the models

In [2]:
# Access drive to get the models
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [56]:
from ultralytics import YOLO

#Load hand segmentation model
hand_model = YOLO('/content/drive/MyDrive/BIAC(TBSA feature)/Hand segmentation model/train/weights/best.pt')

#Load burn segmentation model
burn_model = YOLO('/content/drive/MyDrive/BIAC(TBSA feature)/Burn segmentation model/best (1).pt')

# Get the masks for burns and hand

In [57]:
import torch
from PIL import Image

#get the number of pixels for hand's image
def hand_mask_pixels(hand_model, img):
  #get predictions
  results = hand_model(img, conf=0.5)

  #get the mask
  mask = results[0].masks.data

  #calculate the number of pixels
  num_pixels = torch.count_nonzero(mask)

  #save the result
  results[0].save(filename = 'hand_seg.jpg')

  return num_pixels.item()


#get the number of pixels for burn's images
def burn_mask_pixels(burn_model, imgs):
  #get predictions
  results = burn_model(imgs, conf=0.5)

  #get the masks and calculate the number of pixels
  num_pixels = 0
  for i, result in enumerate(results):
    mask = result.masks.data
    num_pixels = torch.count_nonzero(mask) + num_pixels
    #save the result
    result.save(filename = f'burn{i}.jpg')


  return num_pixels.item()


#images resize
def resize_images(image_paths, new_width, new_height):
    resized_images = []
    for path in image_paths:
        img = Image.open(path)
        img_resized = img.resize((new_width, new_height))
        resized_images.append(img_resized)
    return resized_images

In [58]:
#test of "hand_mask_pixels" function
#load an image
img = Image.open('/content/drive/MyDrive/BIAC(TBSA feature)/TBSA test case with ground truth/Photos/Hand.jpg')

#resize the image
img_resized = img.resize((640, 640))

#calculate the mask pixels
hand_pixels = hand_mask_pixels(hand_model, img_resized)
print("Number of pixels of interest:", hand_pixels)


0: 640x640 1 hand, 565.9ms
Speed: 16.8ms preprocess, 565.9ms inference, 8.1ms postprocess per image at shape (1, 3, 640, 640)
Number of pixels of interest: 48554


In [59]:
#test of "burn_mask_pixels" function
#load image paths
image_pathes = ['/content/drive/MyDrive/BIAC(TBSA feature)/TBSA test case with ground truth/Photos/Burn_1.jpg',
                '/content/drive/MyDrive/BIAC(TBSA feature)/TBSA test case with ground truth/Photos/Burn_2.jpg',
                '/content/drive/MyDrive/BIAC(TBSA feature)/TBSA test case with ground truth/Photos/Burn_3.jpg',
                '/content/drive/MyDrive/BIAC(TBSA feature)/TBSA test case with ground truth/Photos/Burn_4.jpg']

#resize the images
imgs_resized = resize_images(image_pathes, 352, 352)

burn_pixels = burn_mask_pixels(burn_model, imgs_resized)
print("Number of pixels of interest:", burn_pixels)


0: 352x352 1 Degree2, 634.3ms
1: 352x352 1 Degree1, 634.3ms
2: 352x352 1 Degree2, 634.3ms
3: 352x352 1 Degree2, 634.3ms
Speed: 0.9ms preprocess, 634.3ms inference, 2.2ms postprocess per image at shape (1, 3, 352, 352)
Number of pixels of interest: 176523


# Calculate the %TBSA
This equation is different because of the size of hand image is 640x640 and the burn images is 320x320

In [60]:
#calculate the %TBSA by palmer theory(rule of hand)
def calculate_the_tbsa(hand_pixels, burn_pixels):
  #calculate the %TBSA
  tbsa = ((burn_pixels * 3.3) * 0.8) / (hand_pixels)

  return tbsa

In [61]:
#test of "calculate_the_tbsa" function
tbsa = round(calculate_the_tbsa(hand_pixels, burn_pixels), 2)

print(f'%TBSA = {tbsa}%')

%TBSA = 9.6%


# Calculate the fluid resuscitation
the equation that used in this step is parkland formula (4ml * weight * %TBSA)

In [62]:
#calculate the fluid resuscitation
def calculate_the_fluid_amount(weight, tbsa):
  # (4ml * weight * %TBSA)
  fluid_amount = 4 * weight * tbsa

  return fluid_amount

In [63]:
#test of "calculate_the_fluid_amount" function
fluid_amount = calculate_the_fluid_amount(70, tbsa)

print(f'Fluid amount = {fluid_amount} ml')

Fluid amount = 2688.0 ml


#Calculate the survival probability
the equation that used in this step is R-Baux score(Age + %TBSA + 17[Inhalation injury])

In [64]:
#calculate the survival probability
def calculate_the_survival_probability(age, tbsa, inhalation_injury):
  #calculate the R-Baux score
  r_baux_score = age + tbsa + (17 * inhalation_injury)

  #return the survival probability
  if (r_baux_score >= 100):
    r_baux_score = 100
    return 100

  else:
    return 100 - r_baux_score

In [65]:
#test of "calculate_the_survival_probability" function
survival_probability = calculate_the_survival_probability(25, tbsa, 0)

print(f'Survival probability = {survival_probability}%')

Survival probability = 65.4%
